# Random Forests: OOB Validation and Feature Importance Bias

This notebook demonstrates how to use Out-of-Bag (OOB) validation to monitor Random Forest convergence and provides a practical comparison between Mean Decrease in Impurity (Gini Importance) and Permutation Importance under high-cardinality noise.

## 1. Import Dependencies and Generate Credit Risk Dataset

We create a dataset containing:
- `income`: Continuous predictive feature.
- `savings`: Continuous predictive feature.
- `random_noise_id`: A high-cardinality unique identifier (100% random noise) to test feature importance bias.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# Set seed for reproducibility
np.random.seed(42)
m = 600

income = np.random.uniform(20, 200, m)
savings = np.random.uniform(5, 100, m)
# High-cardinality noise column containing unique values
random_noise_id = np.arange(1000, 1000 + m)

# Default if (income + 2 * savings) < 120, else pay
y = np.where((income + 2 * savings) < 120, 1, 0)
# Add 5% noise
y = np.where(np.random.rand(m) < 0.05, 1 - y, y)

df = pd.DataFrame({
    'income': income,
    'savings': savings,
    'random_noise_id': random_noise_id,
    'default': y
})

X = df[['income', 'savings', 'random_noise_id']]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Dataset generated. Features: {list(X.columns)}")

## 2. Monitor Out-of-Bag (OOB) Score Convergence

We sweep the number of estimators (`n_estimators`) from 5 to 150 to track how the OOB score stabilizes.

In [ ]:
estimator_sweep = range(5, 151, 5)
oob_scores = []
test_scores = []

for n in estimator_sweep:
    # Warm start speeds up sequential fitting
    rf = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    oob_scores.append(rf.oob_score_)
    test_scores.append(rf.score(X_test, y_test))

plt.figure(figsize=(8, 4.5))
plt.plot(estimator_sweep, oob_scores, label='OOB Accuracy', color='#339af0', linewidth=2)
plt.plot(estimator_sweep, test_scores, label='Test Accuracy', color='#2b8a3e', linestyle='--', linewidth=1.5)
plt.title("Random Forest OOB score Convergence")
plt.xlabel("Number of Trees (n_estimators)")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

## 3. Diagnose Feature Importance Bias (MDI vs. Permutation Importance)

Let's compare default Scikit-Learn MDI feature importances against Permutation Feature Importance.

In [ ]:
# Fit model with 100 estimators
model_rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

# 1. Extract Gini MDI Importances
mdi_importances = model_rf.feature_importances_

# 2. Calculate Permutation Importances on validation set
perm_results = permutation_importance(
    model_rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
perm_importances = perm_results.importances_mean

# Combine into a DataFrame
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Gini_MDI_Importance': mdi_importances,
    'Permutation_Importance': perm_importances
}).sort_values(by='Gini_MDI_Importance', ascending=False)

print("--- Feature Importance Comparison ---")
print(importance_df.to_string(index=False))

### Plot the results side-by-side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
features = list(importance_df['Feature'])

# Plot Gini Importance
axes[0].barh(features, importance_df['Gini_MDI_Importance'], color=['#e03131', '#adb5bd', '#adb5bd'])
axes[0].set_title("Gini Importance (MDI) - Biased")
axes[0].set_xlabel("Relative Importance Score")
axes[0].invert_yaxis()

# Plot Permutation Importance
axes[1].barh(features, importance_df['Permutation_Importance'], color=['#339af0', '#339af0', '#e03131'])
axes[1].set_title("Permutation Importance (Test Split) - Unbiased")
axes[1].set_xlabel("Relative Drop in Accuracy")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Production Insights
1. **The MDI Trap:** In the left plot, the `random_noise_id` column (which contains completely random unique integers) is ranked as the **most important feature** by Gini MDI. This occurs because the tree splits on this high-cardinality column frequently to fit noise splits on training data.
2. **The Permutation Fix:** In the right plot, Permutation Importance correctly identifies `random_noise_id` as having **0.0 importance** because shuffling it has zero impact on validation predictions. It correctly ranks `savings` and `income` as the true drivers.